# **Main notebook**

### **I. Engineering financial features for time-series stock data**

- Slice the data into our period of interest (2015-01-01 to 2019-12-31).

Obtain: 

**Per stock (daily)**
- Price (close)
- Adj. returns
- Avg. returns over the previous 3 & 7 days 
- SD of adj. closing prices (monthly)
- Volume

**Global (daily)**
- S&P volume
- S&P returns
- VIX (all market return)

**Targets**
- t1: returns from t+1 MO to t+1 MC 
- t8: returns from t+1 MO to t+8 MO
- t7: returns from t+1 MO to t+7 MC

We obtain historical stock prices from the instruction files. ^SPX and ^VIX are from [FRED](https://fred.stlouisfed.org/categories/46). 

In [8]:
import pandas as pd
import numpy as np 

dfSPX = pd.read_csv(f'data/Stock_price/^SPX.csv', sep=';')
dfSPX['date'] = pd.to_datetime(dfSPX['Date'])
dfSPX = dfSPX.set_index('date').sort_index()

dfSPX['dlyretSPX'] = dfSPX['Close'].pct_change()
dfSPX['volumeSPX'] = dfSPX['Volume']
dfSPX = dfSPX.loc['2015-01-01':'2019-12-31', ['dlyretSPX', 'volumeSPX']]

dfVIX = pd.read_csv(f'data/Stock_price/^VIX.csv', sep=';')
dfVIX['date'] = pd.to_datetime(dfVIX['observation_date'])
dfVIX = dfVIX.set_index('date').sort_index()
dfVIX = dfVIX.loc['2015-01-01':'2019-12-31', ['VIXCLS']].rename(columns={'VIXCLS': 'VIX'})

dfGlobal = dfSPX.join(dfVIX, how='inner')

In [9]:
tickers = ("AAPL", "AMZN", "GOOG", "GOOGL", "MSFT", "TSLA")
df = {}

for ticker in tickers:
    
    # Loading data per ticker
    df_temp = pd.read_csv(f'data/Stock_price/{ticker}.csv')
    df_temp['date'] = pd.to_datetime(df_temp['date'])
    df_temp = df_temp.set_index('date').sort_index()
    
    # Filtering time window: 2015-01-01 to 2019-12-31 (margin for returns and SD)
    df_temp = df_temp.loc['2014-12-01':'2020-01-31']


    df_temp['price'] = df_temp['close']
    df_temp['dlyret'] = df_temp['adj close'].pct_change() # Daily adjusted returns
    df_temp['logVolume'] = np.log(df_temp['volume'].clip(lower=1)) 

    df_temp['past_3ret'] = df_temp['dlyret'].rolling(3).mean() # Previous 3 days mean adjusted returns
    df_temp['past_7ret'] = df_temp['dlyret'].rolling(7).mean() # Previous 7 days mean adjusted returns
    df_temp['sdAdjClose'] = df_temp['adj close'].rolling(20).std() # SD of adjusted closing prices, one trading month

    #df_temp['ret1'] = df_temp['dlyret'].shift(-1)  # Forward return from t to t+1
    #df_temp['ret7'] = (df_temp['price'].shift(-7) - df_temp['price']) / df_temp['price'] # Forward return from t to t+7

    df_temp['ret1'] = (df_temp['close'].shift(-1) - df_temp['open'].shift(-1)) / df_temp['open'].shift(-1) 
    df_temp['ret8'] = (df_temp['open'].shift(-8) - df_temp['open'].shift(-1)) / df_temp['open'].shift(-1)
    df_temp['ret7'] = (df_temp['close'].shift(-7) - df_temp['open'].shift(-1)) / df_temp['open'].shift(-1)

    # We compute metrics only for the period of interest
    df_temp = df_temp.loc['2015-01-01':'2019-12-31', ['price', 'volume', 'dlyret', 'sdAdjClose', 'past_3ret', 'past_7ret', 'ret1', 'ret7']]

    # Merging with global features
    df_temp = df_temp.join(dfGlobal, how='left')
    df[ticker] = df_temp

    # Create individual df: dfAAPL exists. Without this line: df['AAPL'] instead
    globals()[f'df{ticker}'] = df[ticker]    

### **II. News headlines**

We scrape the Financial Times pages for each firm of interest.

In [ ]:
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd

SOURCES = {
    "AAPL":     ("apple-inc",                                    39),
    "AMZN":    ("stream/84cf4073-a674-4a93-aef9-dcc1832a65cb", 33),
    "GOOG":  ("stream/6656b3e4-1a78-4960-90a0-8d422450f9a6", 11),
    "GOOGL":    ("stream/d6b12f0c-bf3f-4045-a07b-1e4e49103fd6", 32),
    "MSFT": ("stream/4f447b5d-53f5-41bd-ab42-3c0cfc161699", 25),
    "TSLA":     ("stream/35edec46-ef7b-4f9b-b85a-25174e6e07fa", 32),
}

def scrape_ft_topic(slug, start_page, ticker):
    articles = []
    page = start_page
    while True:
        url = f"https://www.ft.com/{slug}?page={page}"
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        items = soup.select(".stream-item[data-id]")

        if not items:
            print(f"\n  [{ticker}] page {page}: no items, stopping")
            break

        for item in items:
            title_tag = item.select_one(".o-teaser__heading a")
            li = item.find_parent("li")
            time_tag = li.find("time") if li else None
            if not title_tag:
                continue

            date = time_tag["datetime"][:10] if time_tag else ""
            title = title_tag.get_text(strip=True)

            if title.startswith("Lex."):
                title = title[len("Lex."):].strip()

            if date and date[:4] < "2015":
                print(f"\n  [{ticker}] p{page}: reached {date}, stopping")
                return articles

            if date and date[:4] < "2020":
                articles.append({"ticker": ticker, "date": date, "title": title})

        print(f"Scraping {ticker}: Page {page}, {len(articles)} articles", end="\r")
        page += 1
        time.sleep(1)

    return articles


all_articles = []
for ticker, (slug, start_page) in SOURCES.items():
    arts = scrape_ft_topic(slug, start_page, ticker)
    all_articles.extend(arts)

dfHL = pd.DataFrame(all_articles)
dfHL = dfHL.drop_duplicates(subset=["title", "date", "ticker"])
dfHL = dfHL.sort_values(["ticker", "date"], ascending=[True, False])

print(f"\nTotal: {len(dfHL)} articles")
print(dfHL.groupby("ticker").size())
dfHL.to_csv("data/output/HL_FinTimes_raw.csv", index=False)

Scraping AAPL: Page 150, 2776 articles
  [AAPL] p151: reached 2014-12-31, stopping
Scraping AMZN: Page 92, 1491 articles
  [AMZN] p93: reached 2014-12-29, stopping
Scraping GOOG: Page 87, 1912 articles
  [GOOG] p88: reached 2014-12-30, stopping
Scraping GOOGL: Page 53, 517 articles
  [GOOGL] page 54: no items, stopping
Scraping MSFT: Page 56, 793 articles
  [MSFT] p57: reached 2014-12-28, stopping
Scraping TSLA: Page 63, 782 articles
  [TSLA] p64: reached 2014-12-30, stopping

Total: 8307 articles
company
AAPL     2784
AMZN     1488
GOOG     1920
GOOGL     514
MSFT      810
TSLA      791
dtype: int64


We scrape 8308 FT articles headlines: that is less than 1 article per day per firm over the 5-year period we are studying. After cleaning, we keep 3898 articles: the FT includes a lot of articles that aren't relevant. 

As a solution, we scrape other news sources. Instead of designed spiders for each individual news-website (most often would be challening as they have protection against this kind of scraping), we simply leverage Google search engine with such query: "intitle:{keyword} site:www.{site}.com after:2014-12-31 before:2020-01-01".

Using technique, we scrape the following keywords:
- AAPL: Apple, Iphone, Steve Jobs, Tim Cook, iPade, AirPods, MacBook
- AMZN: Amazon, Jeff Bezos, AWS
- GOOG/GOOGL: Google, Sundar Pichai, Alphabet
- MSFT: Satya Nadella, Microsoft
- TSLA: Tesla, Model 3, Elon Musk.


We execute these queries for the following news sources: Reuters, The Guardian, Bloomberg, The New York Times, The Wall Street Journal, CNBC, BBC.

After scraping, we obtain 9011 articles. Cleaning is required. Leverageing special characters, we remove all headlines in foreign languages. As for thematics, we remove headlines mentioning apples (fruits) and the Amazon (forest). =Among the remaning headlines, despite using serious source-news, a lot aren't serious market-moving information, but noisy guides and other consumer-oriented article, among all sources. Hence, we remove headlines mentioning "How...", "review", "games", "which", "tips", "tricks", "best", "app", "guide", "opinion", "What...", "Why...", "How...", etc. We manually execute this last cleaning method: xxxx headlines are remaining.

*We made the choice not to leverage Yahoo Finance. Despite providing market-oriented news only, the quality of certain sources are questionable, often more oriented towards sensational day traders and not institutional quality news.*

We have a total of 11532 headlines.

We scrape 6387 FT articles headlines: that is less than 1 article per day per firm over the 5-year period we are studying. To counter this issue, we scrape other news sources. To avoid building individual scrapers tailored for each website's security measures, we leverage Google search engine with custom queries such as "intitle:"{keyword}" site:www.{site}.com after:2014-12-31 before:2020-01-01".

For keywords, we scrape: 
- AAPL: Apple, iPhone, Steve Jobs, Tim Cook, iPad, AirPods, Macbook
- AMZN: Amazon, Jeff Bezos, AWS
- GOOG/GOOGL: Google, Sundar Pichai, Alphabet
- MSFT/ Satya Nadella, Microsoft
- TSLA: Tesla, Model 3, Elon Musk.

As for news sources, we use Reuters, Bloomberg, The Guardian, The New York Times, The Wall Street Journal, CNBC and BBC.

This method has some limitations: headlines are sometimes truncated, or mention irrelevant topics. Hence, we filter all foreign language headlines, those mentioning the Amazon rainforest or apples as a fruit, as well as many consumer-oriented articles that some of these serious information sources provide.



### **III. Tweeter sentiment analysis**

- Load tweets
- Define influential tweets
- Compute sentiment: VADER, Bag of Words, TextBlob, roBERTa

We first load tweets: they have already been filtered from spam, and finBERT sentiment has been computed.

In [48]:
dfTweets = pd.read_csv(f'data/output/FinBERT_sentiment.csv')

#### **III.A. Influential Tweets**

We then define which tweets are influential. A tweet is defined as influential if any of its metrics (comment_num, retweet_num, like_num) exceed the 99th percentile for those metrics. We distinguish influential tweets from non-influential tweets (SHAO et al., 2025).

In [12]:
com99th, rt99th, like99th = dfTweets['comment_num'].quantile(0.99), dfTweets['retweet_num'].quantile(0.99), dfTweets['like_num'].quantile(0.99)

print(f"99th comments: {com99th}\n99th RT: {rt99th}\n99th like: {like99th}")

# Influential: if any of its metric is above the 99th threshold
dfTweets['influential'] = (
    (dfTweets['comment_num'] > com99th) |
    (dfTweets['retweet_num'] > like99th) |
    (dfTweets['like_num'] > like99th)
)

99th comments: 6.0
99th RT: 11.0
99th like: 45.0


#### **III.B. Sentiment analysis**

We then proceed with various types of sentiment analysis: VADER, TextBlob, Bag of Words, roBERTa. 

##### **III.B.1. VADER**

[VADER](https://github.com/cjhutto/vaderSentiment) sentiment analysis. Instead of using the compound score, we substract the negative sentiment scores from positive sentiment scores as our final variable (SHAO et al., 2025).

We compute VADER scores for both tweets and headlines.

In [19]:
dfFT = pd.read_csv("data/output/HL_FinTimes_cleaned.csv", sep=';')
dfAll = pd.read_csv("data/output/HL_all_cleaned.csv", sep=";")

dfHL = pd.concat([dfFT, dfAll], ignore_index=True)

In [21]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

tweets = dfTweets["body"].fillna('').astype(str)
headlines = dfHL["headline"].fillna('').astype(str)
analyzer = SentimentIntensityAnalyzer()

In [58]:
# 2min
vader_score = []
for tweet in tweets:
    vs = analyzer.polarity_scores(tweet)
    vader_score.append(vs['pos'] - vs['neg'])

dfTweets['VADER'] = vader_score

In [22]:
vader_H_score = []
for headline in headlines:
    vs = analyzer.polarity_scores(headline)
    vader_H_score.append(vs['pos'] - vs['neg'])

dfHL['VADER'] = vader_H_score

##### **III.2.2. Bag of Words (BoW)**

$$\text{BoW score}=
\begin{cases}
\frac{\text{count}_p-\text{count}_n}{\text{count}_p+\text{count}_n}, & \text{if count}_p + \text{count}_n > 0 ;\\
0, & \text{otherwise} ;
\end{cases}
$$

Where $\text{count}_p$ and $\text{count}_n$ denote the count of positive and negative words, respectively.

NLTK initial vocabulary is limited: we add some positive and negative financial jargon to its vocabulary, but the NLKT opinion lexicon still yields discrete sentiment scores (-1, 0, 1) for most tweets due to its limited coverage.

In [23]:
from nltk.corpus import opinion_lexicon
from nltk.tokenize import word_tokenize
import nltk

# Needed ressources
nltk.download('opinion_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_words = set(opinion_lexicon.positive()) # # positive words
neg_words = set(opinion_lexicon.negative())# # negative words

# We add some financial jargon that wasn't in the opinion lexicon
fin_pos = {"surge", "profit", "uptrend", "beat", "buy", "bull", "skyrocket"}
fin_neg = {"plunge", "selloff", "underperform", "downtrend", "sell", "bear", "bankruptcy", "bankrupt"}

pos_words.update(fin_pos)
neg_words.update(fin_neg)

[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/eyquem/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /Users/eyquem/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/eyquem/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [61]:
# 2min (Tweets)
BoW_scores = []
for tweet in tweets:
    tokens = word_tokenize(tweet.lower())
    count_p = sum(1 for token in tokens if token in pos_words)
    count_n = sum(1 for token in tokens if token in neg_words)

    # BoW score formula
    if count_p + count_n > 0: BoW_score = (count_p - count_n) / (count_p + count_n)
    else: BoW_score = 0.0

    BoW_scores.append(BoW_score)

dfTweets['BoW'] = BoW_scores

In [24]:
BoW_H_scores = []

for headline in headlines:
    Htokens = word_tokenize(headline.lower())
    count_p = sum(1 for Htoken in Htokens if Htoken in pos_words)
    count_n = sum(1 for Htoken in Htokens if Htoken in neg_words)

    # BoW score formula
    if count_p + count_n > 0: BoW_H_score = (count_p - count_n) / (count_p + count_n)
    else: BoW_H_score = 0.0

    BoW_H_scores.append(BoW_H_score)

dfHL['BoW'] = BoW_H_scores

##### **III.B.3. TextBlob (TB)**

We use TextBlob for further sentiment analysis. TB alread provides a sentiment polarity score in the range [-1, 1].

In [25]:
from textblob import TextBlob

In [65]:
#3m
TB_scores = []

for tweet in tweets:
    TB = TextBlob(tweet)
    TB_scores.append(TB.sentiment.polarity)

dfTweets['TBlob'] = TB_scores
dfTweets.to_csv("data/output/tweets_preRoBERTa.csv")

In [26]:
TB_H_scores = []

for headline in headlines:
    TB_H = TextBlob(headline)
    TB_H_scores.append(TB_H.sentiment.polarity)

dfHL['TBlob'] = TB_H_scores

##### **III.B.4. RoBERTa**

We use a [pre-trained RoBERTa model fine-tuned for sentiment analysis](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest).

Sentiment of the model: 0: = Negative; 1 = Neutral; 2 = Positive

We access the model via HuggingFace.

In [27]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from scipy.special import softmax
from tqdm import tqdm

# Model config
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

def preprocess(text):
    if not isinstance(text, str):
        return ""
    new_text = []
    for t in text.split(" "):
        new_text.append(t)
    return " ".join(new_text)

def get_RoBERTa_score(text):
    try:
        text = preprocess(text)
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        output = model(**encoded_input)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        
        score = scores[2] - scores[0] # positive - negative
        return score
    except:
        return 0.0

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32508.97it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [67]:
dfTweets = pd.read_csv("data/output/tweets_preRoBERTa.csv")
begin, to = 0, 100
tqdm.pandas(desc="RoBERTa sentiment")
dfTweets['RoBERTa'] = dfTweets['body'].fillna('').astype(str).iloc[begin:to].progress_apply(get_RoBERTa_score)
dfTweets.to_csv(f"data/output/RoBERTa_{begin}_to_{to}.csv", index=True)

RoBERTa sentiment: 100%|██████████| 100/100 [00:03<00:00, 30.03it/s]


In [29]:
tqdm.pandas(desc="RoBERTa sentiment")
dfHL['RoBERTa'] = dfHL['headline'].fillna('').astype(str).progress_apply(get_RoBERTa_score)
dfHL.to_csv(f"data/output/HL_sentiment.csv", index=True)

RoBERTa sentiment:   0%|          | 0/11532 [00:00<?, ?it/s]

RoBERTa sentiment: 100%|██████████| 11532/11532 [06:12<00:00, 30.94it/s]


#### **III.C. Daily weighted sentiment and influence averages**

The paper used these weights:
$$\text{General Avg Weighted by Followers}_z=\frac{\sum^T_{i=1}(F_i+1)\times S_{i,z}}{\sum^T_{i=1}(F_i+1)}$$
$$\text{Tweet Influential}_z=\frac{\sum^T_{i=1}D_i\times S_{i,z}}{\sum^T_{i=1}D_1}$$
$$\text{Tweet Non-Influential}_z=\frac{\sum^T_{i=1}(1-D_i)\times S_{i,z}}{\sum^T_{i=1}(1-D_i)}$$

Where $S_{i,z}$ denotes the sentiment score of the $i$ th tweet for the $z$ th sentiment type (BoW, Vader, TB, or RoBERTa) and $D_i$ denote the binary flag for tweet influence (either 0 or 1) for the $i$ th tweet.

We cannot do weighted averages since we don't have follower counts to define influential users. The rest of the paper mostly use the general average weighted by user per model. We can replicate that doing a global average weighted by influential/non-influential account. We also compute a log-linear weighting, taking into account likes, retweets and comments.

$$\text{General Avg}_z=\frac{1}{T}\sum^T_{i=1}S_{i,z}$$

$$\text{Tweet Influential Weight}_z=\frac{\sum^T_{i=1}(D_i+1)\times S_{i,z}}{\sum^T_{i=1}(D_i+1)}$$

$$\text{Tweet Log-Weighted}_z=\frac{\sum^T_{i=1}\log(1+\text{comments}_i+\text{RT}_i+\text{likes}_i)\times S_{i,z}}{\sum^T_{i=1}\log(1+\text{comments}_i+\text{RT}_i+\text{likes}_i)}$$

For non-trading days, we assign tweets and headlines to the first following trading day, for which we calculate the "daily" average.

We have few days with missing sentiment.

In [10]:
dfTweets = pd.read_csv("data/output/tweets_postRoBERTa.csv")
dfHL = pd.read_csv("data/output/HL_sentiment.csv")

In [4]:
# Assigning all GOOG headlines to GOOGL and all GOOGL headlines to GOOG
g2g = dfHL[dfHL['ticker'] == 'GOOG'].copy()
g2g['ticker'] = 'GOOGL'

gl2gl = dfHL[dfHL['ticker'] == 'GOOGL'].copy()
gl2gl['ticker'] = 'GOOG'

dfHL_symmetrized = pd.concat([
    dfHL,
    g2g,
    gl2gl 
], ignore_index=True)

dfHL = dfHL_symmetrized.drop_duplicates()

#### **Simple average of daily tweets using 3 averaging techniques.**

In [10]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

sentiment_cols = ['VADER', 'BoW', 'TBlob', 'RoBERTa']

for ticker in tickers:
    df_ticker = dfTweets[dfTweets['ticker'] == ticker].copy()
    df_ticker['post_date'] = pd.to_datetime(df_ticker['post_date'], unit='s').dt.normalize().astype('datetime64[us]')


    # We get trading days (normalized, sorted, deduplicated)
    trading_days_df = pd.DataFrame({'date': df[ticker].index.normalize()}).sort_values('date').drop_duplicates()
    trading_days_df['date'] = trading_days_df['date'].astype('datetime64[us]')

    # Fast vectorized assignment: first trading day >= tweet's date
    df_ticker = df_ticker.sort_values('post_date')
    df_ticker['date'] = pd.merge_asof(
        df_ticker[['post_date']],
        trading_days_df,
        left_on='post_date',
        right_on='date',
        direction='forward'
    )['date'].values

    df_ticker = df_ticker.dropna(subset=['date'])

    all_daily = []
    for date, day_df in df_ticker.groupby('date'):
        for sentiment in sentiment_cols:
            all_daily.append({
                'Date': date,
                'Model': sentiment,
                #'General_Avg': day_df[sentiment].mean()
                #'General_Avg': ((day_df['influential'].astype(int) + 1) * day_df[sentiment]).sum() / (day_df['influential'].astype(int) + 1).sum() # Binary influential
                'General_Avg': (np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']) * day_df[sentiment]).sum() / np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']).sum() #log
            })

    daily_df = pd.DataFrame(all_daily).sort_values(['Date', 'Model']).reset_index(drop=True)
    daily_pivot = daily_df.pivot(index='Date', columns='Model', values='General_Avg').reset_index().rename(columns={'Date': 'date'})
    daily_pivot = daily_pivot[['date', 'VADER', 'BoW', 'TBlob', 'RoBERTa']]

    # Merged with individual df
    daily_pivot = daily_pivot.set_index('date')
    df[ticker] = df[ticker].join(daily_pivot, how='left')
    df[ticker] = df[ticker].rename(columns={'VADER': 'TVADER', 'BoW': 'TBoW', 'TBlob': 'TTBlob', 'RoBERTa': 'TRoBERTa'})
    globals()[f'df{ticker}'] = df[ticker]

    print(f"\n{ticker}")
    print(daily_pivot.round(4).head(1))


AAPL
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0198  0.1221  0.0457   0.1651

AMZN
Model        VADER     BoW  TBlob  RoBERTa
date                                      
2015-01-02 -0.0529 -0.2231 -0.217  -0.1145

GOOG
Model        VADER  BoW   TBlob  RoBERTa
date                                    
2015-01-02 -0.0036  0.0  0.0547  -0.0026

GOOGL
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0403 -0.0953  0.1337   0.4622

MSFT
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.2235  0.0465  0.1901   0.3643

TSLA
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0422  0.1036  0.0918   0.1323


- pour prédire t+1, prendre les tweets du soir de t:
      
      - de 16:00 ET après la clôture
      
      - jusqu’à 09:29:59 ET le lendemain
  
  - et les agréger sur le jour t

  Donc:

  - close = 16:00 ET
  - open = 09:30 ET
  - overnight window = [16:00, 09:30)

#### **Daily average = [16:00 to 09:30 d+1), with 3 different techniques.**

In [5]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

sentiment_cols = ['VADER', 'BoW', 'TBlob', 'RoBERTa']
for ticker in tickers:
    df_ticker = dfTweets[dfTweets['ticker'] == ticker].copy()
    df_ticker['post_date'] = pd.to_datetime(df_ticker['post_date'], unit='s', utc=True).dt.tz_convert('America/New_York')

    # keep only overnight tweets: after 16:00 or before 09:30
    h = df_ticker['post_date'].dt.hour + df_ticker['post_date'].dt.minute / 60.0
    df_ticker = df_ticker[(h >= 16.0) | (h < 9.5)].copy()

    # decale de 16h avant normalize : un tweet apres 16h le jour d reste sur d,
    # un tweet avant 9h30 le jour d+1 retombe sur d
    naive = df_ticker['post_date'].dt.tz_localize(None)
    shifted = naive - pd.Timedelta(hours=16)
    df_ticker['assign_date'] = shifted.dt.normalize().astype('datetime64[us]')

    trading_days_df = pd.DataFrame({'date': df[ticker].index.normalize()}).sort_values('date').drop_duplicates()
    trading_days_df['date'] = trading_days_df['date'].astype('datetime64[us]')

    df_ticker = df_ticker.sort_values('assign_date')
    df_ticker['date'] = pd.merge_asof(
        df_ticker[['assign_date']],
        trading_days_df,
        left_on='assign_date',
        right_on='date',
        direction='backward'   # si d lui-meme n'est pas un trading day, recule au dernier jour ouvert
    )['date'].values
    df_ticker = df_ticker.dropna(subset=['date'])

    all_daily = []
    for date, day_df in df_ticker.groupby('date'):
        for sentiment in sentiment_cols:
            all_daily.append({
                'Date': date,
                'Model': sentiment,
                'General_Avg': (np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']) * day_df[sentiment]).sum()
                               / np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']).sum()
            })

    daily_df = pd.DataFrame(all_daily).sort_values(['Date', 'Model']).reset_index(drop=True)
    daily_pivot = daily_df.pivot(index='Date', columns='Model', values='General_Avg').reset_index().rename(columns={'Date': 'date'})
    daily_pivot = daily_pivot[['date', 'VADER', 'BoW', 'TBlob', 'RoBERTa']]

    daily_pivot = daily_pivot.set_index('date')
    df[ticker] = df[ticker].join(daily_pivot, how='left')
    df[ticker] = df[ticker].rename(columns={'VADER': 'TVADER', 'BoW': 'TBoW', 'TBlob': 'TTBlob', 'RoBERTa': 'TRoBERTa'})
    globals()[f'df{ticker}'] = df[ticker]
    print(f"\n{ticker}")
    print(daily_pivot.round(4).head(1))


AAPL
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0482  0.1747  0.0033   0.3157

AMZN
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0749  0.2252  0.1348   0.2821

GOOG
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0829  0.2592  0.1316   0.3628

GOOGL
Model        VADER    BoW   TBlob  RoBERTa
date                                      
2015-01-02  0.0075  0.129 -0.0054   0.3049

MSFT
Model        VADER    BoW   TBlob  RoBERTa
date                                      
2015-01-02  0.1654  0.087  0.1601   0.4256

TSLA
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0388  0.0558  0.0825   0.3228


#### **Daily average: from open t to open t+1**

In [11]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

sentiment_cols = ['VADER', 'BoW', 'TBlob', 'RoBERTa']
for ticker in tickers:
    df_ticker = dfTweets[dfTweets['ticker'] == ticker].copy()
    df_ticker['post_date'] = pd.to_datetime(df_ticker['post_date'], unit='s', utc=True).dt.tz_convert('America/New_York')

    # decale de 9h30 : tout ce qui est avant l'ouverture du jour "tombe" dans la journee precedente
    shifted = df_ticker['post_date'].dt.tz_localize(None) - pd.Timedelta(hours=9, minutes=30)
    df_ticker['assign_date'] = shifted.dt.normalize().astype('datetime64[us]')

    trading_days_df = pd.DataFrame({'date': df[ticker].index.normalize()}).sort_values('date').drop_duplicates()
    trading_days_df['date'] = trading_days_df['date'].astype('datetime64[us]')

    df_ticker = df_ticker.sort_values('assign_date')
    df_ticker['date'] = pd.merge_asof(
        df_ticker[['assign_date']],
        trading_days_df,
        left_on='assign_date',
        right_on='date',
        direction='forward'   # jour futur le plus proche si pas un trading day
    )['date'].values
    df_ticker = df_ticker.dropna(subset=['date'])

    all_daily = []
    for date, day_df in df_ticker.groupby('date'):
        for sentiment in sentiment_cols:
            all_daily.append({
                'Date': date,
                'Model': sentiment,
                'General_Avg': (np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']) * day_df[sentiment]).sum()
                               / np.log1p(day_df['comment_num'] + day_df['retweet_num'] + day_df['like_num']).sum()
            })

    daily_df = pd.DataFrame(all_daily).sort_values(['Date', 'Model']).reset_index(drop=True)
    daily_pivot = daily_df.pivot(index='Date', columns='Model', values='General_Avg').reset_index().rename(columns={'Date': 'date'})
    daily_pivot = daily_pivot[['date', 'VADER', 'BoW', 'TBlob', 'RoBERTa']]

    daily_pivot = daily_pivot.set_index('date')
    df[ticker] = df[ticker].join(daily_pivot, how='left')
    df[ticker] = df[ticker].rename(columns={'VADER': 'TVADER', 'BoW': 'TBoW', 'TBlob': 'TTBlob', 'RoBERTa': 'TRoBERTa'})
    globals()[f'df{ticker}'] = df[ticker]
    print(f"\n{ticker}")
    print(daily_pivot.round(4).head(1))


AAPL
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0247  0.1395  0.0434   0.1866

AMZN
Model       VADER     BoW   TBlob  RoBERTa
date                                      
2015-01-02 -0.038 -0.1698 -0.1775  -0.0869

GOOG
Model        VADER    BoW   TBlob  RoBERTa
date                                      
2015-01-02  0.0076 -0.021  0.0622   0.0422

GOOGL
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0403 -0.0953  0.1337   0.4622

MSFT
Model       VADER     BoW  TBlob  RoBERTa
date                                     
2015-01-02  0.214  0.0436  0.178   0.3665

TSLA
Model        VADER     BoW   TBlob  RoBERTa
date                                       
2015-01-02  0.0359  0.1001  0.0943   0.1431


#### **Assigning daily headlines sentiment**

In [12]:
dfHL['date'] = pd.to_datetime(dfHL['date']).dt.normalize().astype('datetime64[us]')

for ticker in tickers:

    df_ticker = dfHL[dfHL['ticker'] == ticker].copy()

    # We get trading days
    trading_days = df[ticker].index.normalize().astype('datetime64[us]')

    # We assign each tweet and headline to the first following trading day, and drop. those with no upcoming t-day
    df_ticker['date'] = df_ticker['date'].apply(
        lambda x: trading_days[trading_days >= x].min() if len(trading_days[trading_days >= x]) > 0 else None
    )

    df_ticker = df_ticker.dropna(subset=['date'])
    daily_HL = df_ticker.groupby('date')[sentiment_cols].mean()
    daily_HL = daily_HL[['VADER', 'BoW', 'TBlob', 'RoBERTa']]

    df[ticker] = df[ticker].join(daily_HL, how='left')
    df[ticker] = df[ticker].rename(columns={'VADER': 'HLVADER', 'BoW': 'HLBoW', 'TBlob': 'HLTBlob', 'RoBERTa': 'HLRoBERTa'})

    globals()[f'df{ticker}'] = df[ticker]

    print(f"\n{ticker}")
    print(daily_HL.round(4).head(1))

    daily_HL.to_csv("data/output/HL_sentiment_Dly.csv")


AAPL
            VADER  BoW  TBlob  RoBERTa
date                                  
2015-01-02 -0.096 -0.5    0.0  -0.2903

AMZN
            VADER  BoW  TBlob  RoBERTa
date                                  
2015-01-12  0.254  1.0    0.3   0.4691

GOOG
            VADER  BoW  TBlob  RoBERTa
date                                  
2015-01-05    0.0  0.5    0.0   0.2489

GOOGL
             VADER  BoW  TBlob  RoBERTa
date                                   
2015-08-10  0.1238  0.5  0.026   0.2921

MSFT
            VADER  BoW  TBlob  RoBERTa
date                                  
2015-01-12    0.0 -1.0    0.0  -0.6122

TSLA
            VADER  BoW  TBlob  RoBERTa
date                                  
2015-01-06    0.0  0.0    0.0   0.2874


In [13]:
for ticker in tickers:
    total_days = len(df[ticker])
    pct_missing_tweets = df[ticker]['TVADER'].isna().mean() * 100
    pct_missing_hl = df[ticker]['HLVADER'].isna().mean() * 100
    print(f"{ticker}: {pct_missing_tweets:.1f}% no T | {pct_missing_hl:.1f}% no HL")
    na_mask = df[ticker]['HLVADER'].isna()
    groups = (na_mask != na_mask.shift()).cumsum()
    seq_lengths = na_mask.groupby(groups).sum()
    seq_lengths = seq_lengths[seq_lengths > 0]
    print(f"{ticker}: max consecutive missing values = {seq_lengths.max()} days | med = {seq_lengths.median()} | missing > 10d = {(seq_lengths > 10).sum()}")

AAPL: 0.1% no T | 14.8% no HL
AAPL: max consecutive missing values = 4 days | med = 1.0 | missing > 10d = 0
AMZN: 0.0% no T | 29.3% no HL
AMZN: max consecutive missing values = 10 days | med = 1.0 | missing > 10d = 0
GOOG: 0.0% no T | 25.8% no HL
GOOG: max consecutive missing values = 6 days | med = 1.0 | missing > 10d = 0
GOOGL: 0.0% no T | 72.7% no HL
GOOGL: max consecutive missing values = 151 days | med = 2.0 | missing > 10d = 17
MSFT: 0.0% no T | 54.8% no HL
MSFT: max consecutive missing values = 11 days | med = 2.0 | missing > 10d = 3
TSLA: 0.0% no T | 40.5% no HL
TSLA: max consecutive missing values = 13 days | med = 1.0 | missing > 10d = 2


In [14]:
dfAAPL.to_csv("data/training/AAPL_open_open_AVG3.csv")
dfAMZN.to_csv("data/training/AMZN_open_open_AVG3.csv")
dfGOOG.to_csv("data/training/GOOG_open_open_AVG3.csv")
dfGOOGL.to_csv("data/training/GOOGL_open_open_AVG3.csv")
dfMSFT.to_csv("data/training/MSFT_open_open_AVG3.csv")
dfTSLA.to_csv("data/training/TSLA_open_open_AVG3.csv")

We now have every feature for every day of every asset. We save each way of computing the average in order not to compute it again for the model. 

We aren't missing a lot of tweets. However, a lot of days are missing headlines: up to 54% of days for Microsoft, and sequences of up to 13 days of missing values. However, the median remains at 1 or 2 days with no headline sentiment. We need to select a filling methodology:
- using the last value
- using the average over the past x values
- using the last value with a decay factor
- using an exponentially weighted average

For the final model, we decided after comparison to use an EWMA on the past 5 values, with a 0.5 decay factor. 

As for the average, the log-weighted version on tweet metrics is the one that performs the best.

For tweets, we tried three different methods of daily attribution:
- simple daily: a tweet in window [00:00, 23:59) is attributed to day d
- overnight sentiment: a tweet in window [16:00 d, 9:29 d+1) is attributed to day d
- open to open: a tweet in window [9:30 d, 9:29 d+1) is attributed to day d


For all of these methods, we attribute sentiment of non-trading days to the closest future trading day. We find that method xxx performs best when comparing on equivalent, tweet only, models. Generally, overnight models perform slightly better. Open-open doesn't have any big difference to the normal daily model.

Good performers in R^2 include: 
- overnight VADER
- overnight TBlob VADER

Good performers in hit-rate include:
- overnight V2 Bow (52%)
- overnight V2 BoW VADER

### **III. Methods & models**

#### **III.A. HD-SURDLM (Heterogeneous Dynamic Seemingly Unrelated Regression with Dynamic Linear Models) model**

Pros of this model: 

- takes into account multi-dimensional time-varying features
- cross-sectional stock analysis: correlations and spillover over assets, better than single-stock models

The extended model is articulated as: 

$$\begin{align}
&y_{j,t}=\alpha_J+(\sum^K_{k=1}X_{j,t,k}\beta_{j,t,k}) + Z_{j,t}\gamma_j / u_{j,t} \\
&\beta_{j,t,k}=\beta_{j,t-1,k}+v_{j,t,k}
\end{align}
$$

where 
- $j= 1,... , M$ denotes individual stocks
- $t = 1,... , T$ represents discrete time points
- $k= 1,... , K$ indexes over the time-varying features. 

It comprises:
- stock-specific intercepts $\alpha_j$
- time-varying features $X_{j, t, k}$ with coefficients $\beta_{j,t,k}$
- non-time-varying features $Z_{j,t}$ with coefficients $\gamma_j$. 

The model accounts for observation errors $u_{j,t} \sim \mathcal{N}_M(0, \Omega)$ and system errors $v_{j,t,k} \sim \mathcal{N}(0, \Omega)$, which are independent across time and features, with a covariance structure $\sum = A\Omega A$. 

The unknown parameters $\Theta = \{\alpha_j,\beta_{j,t,k},\gamma_j,\Omega\}_{j=1,...,M;t=1,...,T}$ collectively capture stock-specific effects, time-varying relationships, and error structures, enhancing the model’s adaptability to complex, dynamic stock return patterns across multiple assets. 

The model utilizes collected data $D=\{ y_{j,t},X_{j,t,k}, Z_{j,t}\}$, enabling a comprehensive analysis of multi-asset stock returns while accounting for both time-varying and static influences.


 
- $\alpha_j$ serves as the intercept for each stock $j$, epitomizing the intrisinc level of returns in the absence of influence by other variables. 

- The coefficients $\beta_{j,t,k}$ are pivotal, representing the time-varying impact of the $k$ th predictor $X_{j,t,k}$ on the returns of stock $j$ at time $t$ (they may evolve as per $\beta_{j,t,k}=\beta_{j,t-1,k}+v_{j,t,k}$), portraying the temporal variations in the relationships between predictors and returns.

- The model incorporate $\gamma_j$, the set of coefficients corresponding to the predictor vector Z_{j,t} for each stock $j$. Although these coefficient are static across time, their random nature allows fot th emodulation of the the predictors’ impact on returns.

- $\Omega$, encompassed within the unknown parameter vector $\Theta$, is not explicitly defined in the provided context but is potentially indicative of the variance–covariance matrix of the error terms, thereby capturing the variances and correlations in the model residuals.

- The model accommodates error terms representing the observation error and system error, respectively. $u_{j,t}$ accounts for the unexplained variability in stock returns. $v_{j,t,k}$ signifies the fluctuations in the time-varying regression coefficients, thereby contributing to the dynamic nature of the model. 

Collectively, the incorporation of these random parameters enhances the model’s adaptability and proficiency in capturing the multifaceted relationships between various predictors and stock returns across diverse stocks and temporal instances.




We begin by building the feature matrix. For both t+1 and t+7 targets, we use t sentiment and financial metrics.
- t+1 : target is return from t+1 MO to t+1 MC
- t+7: target is return from t+1 MO to t+8 MO
- t+7 alternative: target is return from t+1 MO to t+7 MC

**For sentiment features**
- Model 1: General daily average
- Model 2: General daily weighted average for Tweets
- Model 3: Twitter uses VADER, news uses TB
- Model 4: RoBERTa is used for all
- Can test only Tweet/ only headlines, with each method, or 2 combination, or 3 combinations.

#### **III.B. Stabilized Gibbs samplings**

*Gibbs sampling is a Markov chain Monte Carlo (MCMC) algorithm used for sampling from a multivariate probability distribution when direct sampling is difficult, by sampling from the conditional distributions of each variable.*

We introduce a novel, stable Gibbs sampling approach that not only enhances performance but also effectively handles multiple assets, uncovering potential inter-asset relationships.

*Sampling $\alpha$ and $\gamma$*. We sample the intercept vector $\alpha=(\alpha_1,...,\alpha_m)$ and the non-time-varying coefficient matric $\gamma=(\gamma_1,...,\gamma_M)$:

$(\alpha | D, \Theta \setminus \{\alpha\}) \sim \mathcal{N}(\mu_{\alpha}, \Sigma_{\alpha})$

$(\gamma | D, \Theta \setminus \{\gamma\}) \sim \mathcal{N}_{M \times p}(\mu_{\gamma}, \Sigma_{\gamma})$

where $D$ represents the data, $\Theta$ is the set of all parameters, and $p$ is the number of non-time-varying predictors. $\mu_{\alpha}$, $\Sigma_{\alpha}$, $\mu_{\gamma}$, and $\Sigma_{\gamma}$ are derived from the full conditional distributions.

Sampling $\beta$. We sample the time-varying coefficient tensors $\beta = (\beta_{1,t,k}, \dots, \beta_{M,t,k})$ for each time-varying feature $k = 1, \dots, K$:

$(\beta_{\cdot,\cdot,k} | D, \Theta \setminus \{\beta_{\cdot,\cdot,k}\}) \sim \mathcal{N}_{M \times T}(\mathbf{H}_k, \mathbf{B}_k)$

We use an improved Filter Forward Backward Sampling (FFBS) algorithm that processes all M stocks concurrently:
1. Forward filtration: Compute $\mathbf{m}_t$ and $\mathbf{C}_t$ sequentially for $t = 1, \dots, T$.
2. Backward sampling: For $t = T, \dots, 1$, sample
$\boldsymbol{\beta}_t \sim \mathcal{N}_M(\mathbf{h}_t, \mathbf{B}_t),$
where:

$\mathbf{h}_t = \mathbf{m}_t + \mathbf{C}_t \mathbf{R}_{t+1}^{-1} (\mathbf{m}_{t+1} - \mathbf{m}_t),$

$\mathbf{B}_t = \mathbf{C}_t + \mathbf{C}_t \mathbf{R}_{t+1}^{-1} \mathbf{B}_{t+1} \mathbf{R}_{t+1}^{-1} \mathbf{C}_t - \mathbf{C}_t \mathbf{R}_{t+1}^{-1} \mathbf{C}_t.$

This modification enhances numerical stability by using the computed mean $\mathbf{m}_{t+1}$ instead of the randomly sampled $\boldsymbol{\beta}_{t+1}$, reducing the accumulation of randomness in the backward sampling step.

Sampling $\boldsymbol{\Omega}$. We sample the joint variance–covariance matrix $\boldsymbol{\Omega}$, which captures cross-stock error correlations:
$(\boldsymbol{\Omega} | D \setminus \{\boldsymbol{\beta}\}, \Theta \setminus \{\boldsymbol{\Omega}\}) \sim \mathcal{IW}^{-1}(\nu, \boldsymbol{\Psi}),$

where:

$\nu = T - M + 2,$

$$\boldsymbol{\Psi} = \sum_{t=1}^{T} \mathbf{u}_t \mathbf{u}_t^\top. \tag{17}$$

We do not incorporate $\mathbf{v}_t$ (the difference between consecutive $\boldsymbol{\beta}_t$) in the sampling process. Instead, we recommend estimating $\boldsymbol{\Sigma}$ after the Gibbs sampling to ensure stability and prevent potential "viscous cycles" that could hinder convergence.

This novel approach allows for a more stable estimation of the overall model parameters, preventing situations where errors in estimating $\boldsymbol{\Omega}$ and $\boldsymbol{\beta}$ can reinforce each other, leading to poor convergence or instability in the Gibbs sampler.

Multi-step forecasting. We introduce a flexible approach to multi-step forecasting that accounts for increasing uncertainty across all M stocks simultaneously:

$\mathbf{Y}_{T+N} \sim \mathcal{N}_M(\boldsymbol{\mu}_{T+N}, \boldsymbol{\Sigma}_{T+N}),$

$\boldsymbol{\mu}_{T+N} = \boldsymbol{\alpha} + \mathbf{X}_{T+N} \mathbf{m}_T + \mathbf{Z}_{T+N} \boldsymbol{\gamma},$

$\boldsymbol{\Sigma}_{T+N} = \mathbf{F}_{T+N} (\mathbf{C}_N + T \boldsymbol{\Sigma}) \mathbf{F}_{T+N}^\top + \boldsymbol{\Omega}.$

Here, $\mathbf{Y}_{T+N}$ is an M-dimensional vector of stock returns, $\boldsymbol{\alpha}$ is the M-dimensional vector of intercepts, $\mathbf{X}_{T+N}$ and $\mathbf{Z}_{T+N}$ are matrices of time-varying and non-time-varying predictors for all M stocks, $\mathbf{m}_T$ is the matrix of final state estimates for the time-varying coefficients, and $\boldsymbol{\gamma}$ is the matrix of non-time-varying coefficients. $\mathbf{F}_{T+N}$ is a block-diagonal matrix of the time-varying predictors, $\mathbf{C}_N$ is the covariance matrix of the state estimates, and $\boldsymbol{\Omega}$ is the $M \times M$ cross-sectional covariance matrix of the errors.

This formulation allows for a more accurate representation of forecast uncertainty as the prediction horizon ($N$) increases while also capturing cross-sectional dependencies between stocks. The covariance matrix $\boldsymbol{\Sigma}_{T+N}$ grows with $N$, reflecting the increased uncertainty in longer-term forecasts, and its off-diagonal elements represent the co-movement of stock returns in future predictions.


### **Version numba, working**

In [76]:
# First batch

dfHL = pd.read_csv("data/output/HL_sentiment_Dly.csv")

dfAAPL = pd.read_csv("data/training/AAPL_AVG1.csv", index_col=0, parse_dates=True)
dfAMZN = pd.read_csv("data/training/AMZN_AVG1.csv", index_col=0, parse_dates=True)
dfGOOG = pd.read_csv("data/training/GOOG_AVG1.csv", index_col=0, parse_dates=True)
dfGOOGL = pd.read_csv("data/training/GOOGL_AVG1.csv", index_col=0, parse_dates=True)
dfMSFT = pd.read_csv("data/training/MSFT_AVG1.csv", index_col=0, parse_dates=True)
dfTSLA = pd.read_csv("data/training/TSLA_AVG1.csv", index_col=0, parse_dates=True)

df = {"AAPL": dfAAPL, "AMZN": dfAMZN, "GOOG": dfGOOG, "GOOGL": dfGOOGL, "MSFT": dfMSFT, "TSLA": dfTSLA}

In [83]:
import numpy as np
import pandas as pd
from scipy.stats import invwishart
from numba import njit

########## CONFIG ##########
TICKERS       = ["AAPL", "AMZN", "GOOG", "GOOGL", "MSFT", "TSLA"]
#TV_FEATURES   = ["TVADER", "HLTBlob"]                          # X : time-varying (sentiment)
TV_FEATURES   = ["TRoBERTa", "HLRoBERTa"] 
STAT_FEATURES = ["dlyret", "past_3ret", "past_7ret",
                 "volumeSPX", "dlyretSPX", "VIX"]              # Z : non-time-varying
TARGETS       = ["ret1", "ret7"]

N_ITER  = 5000 # Paper: 5000
BURN_IN = 1000  # Paper: 1000
DELTA   = 0.9   # signal-to-noise ratio. Paper: 0.9


########### STEP 1 — BUILD ALIGNED PANEL ##########
def build_panel(df: dict, target: str, max_T: int = None):
    """
    Returns Y (T×M), X (T×M×K), Z (T×M×p) aligned on common trading dates.
    NaNs in sentiment are forward-filled then zero-filled.
    """
    M = len(TICKERS)
    K = len(TV_FEATURES)
    p = len(STAT_FEATURES)

    common_idx = df[TICKERS[0]].index
    for t in TICKERS[1:]:
        common_idx = common_idx.intersection(df[t].index)
    common_idx = common_idx.sort_values()
    if max_T is not None:
        common_idx = common_idx[:max_T]
    T = len(common_idx)

    Y = np.full((T, M), np.nan)
    X = np.full((T, M, K), np.nan)
    Z = np.full((T, M, p), np.nan)

    for j, ticker in enumerate(TICKERS):
        dft = df[ticker].reindex(common_idx)
        dft[TV_FEATURES]   = dft[TV_FEATURES].ffill().fillna(0)
        dft[STAT_FEATURES] = dft[STAT_FEATURES].ffill().fillna(0)
        Y[:, j]    = dft[target].values
        X[:, j, :] = dft[TV_FEATURES].values
        Z[:, j, :] = dft[STAT_FEATURES].values

    valid = ~np.isnan(Y).any(axis=1)
    return Y[valid], X[valid], Z[valid], common_idx[valid]


# =============================================================================
# NUMBA-ACCELERATED KERNELS
# These functions are JIT-compiled — called inside run_gibbs.
# Numba does not support scipy, so only numpy ops are used here.
# =============================================================================

@njit(cache=True)
def _forward_filter(r_k, Xk, Omega, W, sigma2_v, delta, T, M):
    """
    Forward Kalman filter for one feature k.
    r_k  : (T, M)  partial residual
    Xk   : (T, M)  feature k values across all stocks
    Returns m (T,M), C (T,M,M), m_pred (T,M), R_pred (T,M,M)
    """
    m      = np.zeros((T, M))
    C      = np.zeros((T, M, M))
    m_pred = np.zeros((T, M))
    R_pred = np.zeros((T, M, M))

    # Diffuse initialisation
    init_scale = sigma2_v / (1.0 - delta + 1e-10)
    for i in range(M):
        for j in range(M):
            R_pred[0, i, j] = init_scale * Omega[i, j]

    for t in range(T):
        # F_t = diag(Xk[t, :])
        # f_t = F_t @ m_pred[t]
        f_t = np.zeros(M)
        for i in range(M):
            f_t[i] = Xk[t, i] * m_pred[t, i]

        # Q_t = F_t @ R_pred[t] @ F_t.T + Omega
        Q_t = np.zeros((M, M))
        for i in range(M):
            for j in range(M):
                Q_t[i, j] = Xk[t, i] * R_pred[t, i, j] * Xk[t, j] + Omega[i, j]
        # Symmetrise + regularise
        for i in range(M):
            for j in range(M):
                Q_t[i, j] = 0.5 * (Q_t[i, j] + Q_t[j, i])
            Q_t[i, i] += 1e-8

        # Innovation
        e_t = np.zeros(M)
        for i in range(M):
            e_t[i] = r_k[t, i] - f_t[i]

        # Kalman gain: A_t = R_pred[t] @ F_t.T @ Q_inv
        # First: R_pred[t] @ F_t.T  (F_t is diagonal => F_t.T = F_t)
        RFt = np.zeros((M, M))
        for i in range(M):
            for j in range(M):
                RFt[i, j] = R_pred[t, i, j] * Xk[t, j]

        # Q_inv via Gaussian elimination (small M=6, fast)
        Q_inv = np.linalg.inv(Q_t)
        A_t = RFt @ Q_inv   # (M, M)

        # Update: m[t] = m_pred[t] + A_t @ e_t
        Ae = A_t @ e_t
        for i in range(M):
            m[t, i] = m_pred[t, i] + Ae[i]

        # C[t] = R_pred[t] - A_t @ F_t @ R_pred[t]
        # A_t @ F_t: A_t col j * Xk[t,j]
        AFR = np.zeros((M, M))
        for i in range(M):
            for j in range(M):
                for l in range(M):
                    AFR[i, j] += A_t[i, l] * Xk[t, l] * R_pred[t, l, j]
        for i in range(M):
            for j in range(M):
                C[t, i, j] = R_pred[t, i, j] - AFR[i, j]
        # Symmetrise + regularise
        for i in range(M):
            for j in range(M):
                C[t, i, j] = 0.5 * (C[t, i, j] + C[t, j, i])
            C[t, i, i] += 1e-8

        # Predict next state
        if t < T - 1:
            for i in range(M):
                m_pred[t+1, i] = m[t, i]
            for i in range(M):
                for j in range(M):
                    R_pred[t+1, i, j] = 0.5 * (C[t, i, j] + C[t, j, i]) + W[i, j]

    return m, C, m_pred, R_pred


@njit(cache=True)
def _backward_sample_means(m, C, m_pred, R_pred, T, M):
    """
    Stabilised backward pass — returns h (T,M) and B (T,M,M).
    Uses m[t+1] instead of sampled beta[t+1] (paper stabilisation).
    Caller draws samples from N(h[t], B[t]).
    """
    h = np.zeros((T, M))
    B = np.zeros((T, M, M))

    # Last step
    for i in range(M):
        h[T-1, i] = m[T-1, i]
    for i in range(M):
        for j in range(M):
            B[T-1, i, j] = C[T-1, i, j]

    B_next = np.zeros((M, M))
    for i in range(M):
        for j in range(M):
            B_next[i, j] = C[T-1, i, j]

    for t in range(T-2, -1, -1):
        R_next = np.zeros((M, M))
        for i in range(M):
            for j in range(M):
                R_next[i, j] = R_pred[t+1, i, j] + 1e-8 * (1.0 if i == j else 0.0)

        R_next_inv = np.linalg.inv(R_next)

        # CR = C[t] @ R_next_inv
        CR = C[t] @ R_next_inv   # (M, M)

        # h[t] = m[t] + CR @ (m[t+1] - m[t])
        diff = np.zeros(M)
        for i in range(M):
            diff[i] = m[t+1, i] - m[t, i]
        CR_diff = CR @ diff
        for i in range(M):
            h[t, i] = m[t, i] + CR_diff[i]

        # B[t] = C[t] + CR @ B_next @ CR.T - CR @ C[t]
        CR_Bnext = CR @ B_next          # (M, M)
        CR_Bnext_CRt = CR_Bnext @ CR.T  # (M, M)
        CR_Ct = CR @ C[t]               # (M, M)
        for i in range(M):
            for j in range(M):
                B[t, i, j] = C[t, i, j] + CR_Bnext_CRt[i, j] - CR_Ct[i, j]
        # Symmetrise + regularise
        for i in range(M):
            for j in range(M):
                B[t, i, j] = 0.5 * (B[t, i, j] + B[t, j, i])
            B[t, i, i] += 1e-8

        # Carry B[t] as B_next for next iteration
        for i in range(M):
            for j in range(M):
                B_next[i, j] = B[t, i, j]

    return h, B


########## STEP 2 — GIBBS SAMPLER ##########
def run_gibbs(Y, X, Z, n_iter=N_ITER, burn_in=BURN_IN, delta=DELTA, seed=42):
    """
    HD-SURDLM Gibbs sampler — paper-faithful implementation with Numba acceleration.

    Model
    -----
    y_{j,t} = alpha_j + sum_k X_{j,t,k} * beta_{j,t,k} + Z_{j,t} @ gamma_j + u_{j,t}
    beta_{j,t,k} = beta_{j,t-1,k} + v_{j,t,k}

    u_t ~ N_M(0, Omega)   (cross-stock correlated observation errors)
    v_{t,k} ~ N(0, Omega) (system errors share the same covariance structure)

    FFBS is run in full M-dimensional form for each feature k, so that
    cross-stock correlations in Omega propagate into beta sampling.
    The forward filter and backward pass are JIT-compiled via Numba.

    Parameters
    ----------
    Y : (T, M)
    X : (T, M, K)
    Z : (T, M, p)
    delta : signal-to-noise ratio (0.9 per paper)
    """
    rng   = np.random.default_rng(seed)
    T, M  = Y.shape
    K     = X.shape[2]
    p     = Z.shape[2]

    # ---- Initialisation ----
    alpha = np.zeros(M)                 # (M,)
    gamma = np.zeros((M, p))            # (M, p)
    beta  = np.zeros((T, M, K))        # (T, M, K)
    Omega = np.eye(M)                  # (M, M)  cross-stock covariance

    # System noise covariance: Sigma = A @ Omega @ A  with A = sqrt((1-delta)/delta) * I
    # => sigma2_v scalar; W_k = sigma2_v * Omega  for each feature k
    sigma2_v  = (1.0 - delta) / delta  # scalar factor

    # Storage (post burn-in)
    n_samples     = n_iter - burn_in
    alpha_samples = np.zeros((n_samples, M))
    gamma_samples = np.zeros((n_samples, M, p))
    beta_samples  = np.zeros((n_samples, T, M, K))
    Omega_samples = np.zeros((n_samples, M, M))

    def compute_residuals(alpha_, beta_, gamma_):
        """u_{t} = y_t - alpha - X_t * beta_t - Z_t @ gamma   shape (T, M)"""
        sent   = np.einsum('tmk,tmk->tm', X, beta_)      # (T, M)
        static = np.einsum('tmp,mp->tm', Z, gamma_)       # (T, M)
        return Y - alpha_[None, :] - sent - static

    # Warm up Numba JIT on first call (compiles once, cached after)
    print("  [Numba] Compiling JIT kernels on first iteration...")

    for it in range(n_iter):

        Omega_inv = np.linalg.inv(Omega)

        # 1. Sample alpha | rest  →  N_M(mu_alpha, Sigma_alpha)
        #    Full conditional uses Omega for cross-stock correlation
        r_alpha = (Y
                   - np.einsum('tmk,tmk->tm', X, beta)
                   - np.einsum('tmp,mp->tm', Z, gamma))   # (T, M)

        # Sigma_alpha_inv = T * Omega_inv  (flat prior => prior term ~ 0)
        Sigma_alpha = np.linalg.inv(T * Omega_inv)
        mu_alpha    = Sigma_alpha @ (Omega_inv @ r_alpha.sum(axis=0))
        alpha       = rng.multivariate_normal(mu_alpha, Sigma_alpha)

        # 2. Sample gamma | rest  →  N_{M×p}(mu_gamma, Sigma_gamma)
        #    Per-stock GLS using full Omega diagonal (off-diag ignored in
        #    gamma because Z is stock-specific; cross-terms enter via Omega)
        r_gamma = (Y
                   - alpha[None, :]
                   - np.einsum('tmk,tmk->tm', X, beta))   # (T, M)

        for j in range(M):
            Zj  = Z[:, j, :]        # (T, p)
            rj  = r_gamma[:, j]     # (T,)
            # Use full row of Omega_inv for cross-stock weighting
            # Simplified: use diagonal element (exact for independent stocks)
            sig_jj = Omega[j, j]
            Sig_gj_inv = Zj.T @ Zj / sig_jj   # flat prior
            Sig_gj     = np.linalg.inv(Sig_gj_inv + 1e-8 * np.eye(p))
            mu_gj      = Sig_gj @ (Zj.T @ rj) / sig_jj
            gamma[j]   = rng.multivariate_normal(mu_gj, Sig_gj)

        # 3. Sample beta via M-dimensional FFBS  (paper Section III.B)
        #    For each feature k, beta_{:,k} is an (M,)-vector evolving over t.
        #
        #    Observation eq:  r_{t,k} = diag(X_{t,:,k}) @ beta_{t,k} + u_t
        #    State eq:        beta_{t,k} = beta_{t-1,k} + v_{t,k}
        #
        #    u_t  ~ N_M(0, Omega)
        #    v_{t,k} ~ N_M(0, W_k)  with W_k = sigma2_v * Omega
        #
        #    Forward filter and backward pass are Numba-compiled.
        r_beta = (Y
                  - alpha[None, :]
                  - np.einsum('tmp,mp->tm', Z, gamma))    # (T, M)

        W = sigma2_v * Omega    # system noise covariance (M, M)

        for k in range(K):
            # Partial residual: remove other features' contributions
            r_k = r_beta.copy()                           # (T, M)
            for kk in range(K):
                if kk != k:
                    r_k -= X[:, :, kk] * beta[:, :, kk]

            Xk = X[:, :, k]   # (T, M)

            # --- Numba forward filter ---
            m, C, m_pred, R_pred = _forward_filter(
                r_k, Xk, Omega, W, sigma2_v, delta, T, M
            )

            # --- Numba backward pass (returns h, B for each t) ---
            h, B = _backward_sample_means(m, C, m_pred, R_pred, T, M)

            # --- Draw samples (numpy, not Numba — requires RNG) ---
            b = np.zeros((T, M))
            for t in range(T):
                b[t] = rng.multivariate_normal(h[t], B[t])

            beta[:, :, k] = b   # (T, M)

        # 4. Sample Omega | residuals  →  IW(nu, Psi)
        #    nu = T - M + 2   (paper eq. 16)
        #    Psi = sum_t u_t u_t^T   (paper eq. 17)
        #    Note: v_t NOT included (paper recommendation for stability)
        U   = compute_residuals(alpha, beta, gamma)       # (T, M)
        Psi = U.T @ U                                     # (M, M)
        nu  = max(T - M + 2, M + 2)                      # ensure nu > M-1
        Psi = (Psi + Psi.T) / 2 + 1e-6 * np.eye(M)
        Omega = invwishart.rvs(df=nu, scale=Psi, random_state=rng)

        # Store post burn-in
        if it >= burn_in:
            s = it - burn_in
            alpha_samples[s] = alpha
            gamma_samples[s] = gamma
            beta_samples[s]  = beta.copy()
            Omega_samples[s] = Omega

        if (it + 1) % 500 == 0:
            print(f"  Iteration {it+1}/{n_iter}")

    return {
        "alpha": alpha_samples,   # (n_samples, M)
        "gamma": gamma_samples,   # (n_samples, M, p)
        "beta":  beta_samples,    # (n_samples, T, M, K)
        "Omega": Omega_samples,   # (n_samples, M, M)
    }


########## STEP 3 — GELMAN-RUBIN DIAGNOSTIC ##########
def gelman_rubin(chains: list) -> float:
    """
    chains : list of 1-D arrays (one per chain), post burn-in samples.
    Returns R-hat. Convergence: R-hat < 1.1
    """
    n      = min(len(c) for c in chains)
    chains = [c[:n] for c in chains]
    m      = len(chains)
    theta_bar  = np.array([c.mean() for c in chains])
    grand_mean = theta_bar.mean()
    B     = n / (m - 1) * np.sum((theta_bar - grand_mean)**2)
    W     = np.mean([c.var(ddof=1) for c in chains])
    V_hat = (n - 1) / n * W + (m + 1) / (m * n) * B
    return np.sqrt(V_hat / (W + 1e-12))

def estimate_Sigma_post_gibbs(beta_samples_list):
    """
    Estime Σ après le Gibbs sampling (section 4.2, papier) :
    "we recommend estimating Σ after the Gibbs sampling to ensure stability".

    Utilise les incréments v_t = beta_t - beta_{t-1} (éq. A.22), calculés
    sur CHAQUE échantillon post burn-in de TOUTES les chaînes combinées.
    Version vectorisée (pas de boucle Python sur s/k).

    Parameters
    ----------
    beta_samples_list : list of arrays, chacun (n_samples, T, M, K)

    Returns
    -------
    Sigma_hat : (M, M)
    """
    M = beta_samples_list[0].shape[2]

    Psi_Sigma = np.zeros((M, M))
    n_total   = 0

    for beta_samples in beta_samples_list:
        n_samples, T, M_, K = beta_samples.shape

        # v: (n_samples, T-1, M, K) — incréments sur l'axe temps
        v = beta_samples[:, 1:, :, :] - beta_samples[:, :-1, :, :]

        # Réorganiser pour traiter (n_samples * (T-1) * K) comme des observations (M,)
        # v shape (n_samples, T-1, M, K) -> (n_samples, T-1, K, M) -> (n_samples*(T-1)*K, M)
        v = np.moveaxis(v, 2, -1)                      # (n_samples, T-1, K, M)
        V = v.reshape(-1, M)                            # (n_samples*(T-1)*K, M)

        Psi_Sigma += V.T @ V
        n_total   += V.shape[0]

    nu_Sigma  = n_total - M + 2          # par analogie avec nu = T - M + 2 (éq. 16)
    Psi_Sigma = (Psi_Sigma + Psi_Sigma.T) / 2 + 1e-6 * np.eye(M)

    Sigma_hat = Psi_Sigma / (nu_Sigma - M - 1)
    return Sigma_hat


########### STEP 4 — FORECAST  (paper multi-step forecasting) ##########
def forecast(chains: dict, X_new: np.ndarray, Z_new: np.ndarray, Sigma_hat: np.ndarray, N: int = 1):
    """
    Multi-step forecast à l'horizon N (papier, éq. 18-20).

    Sigma_hat : (M, M) — estimée séparément après le Gibbs sampling
                (cf. estimate_Sigma_post_gibbs), PAS dérivée de sigma2_v*Omega.

    Parameters
    ----------
    X_new : (M, K)   sentiment features à T+N
    Z_new : (M, p)   features financières à T+N
    N     : horizon de prévision (1 pour ret1, 7 pour ret7)

    Returns
    -------
    mu    : (M,)    prévision moyenne posterior par stock
    Sigma : (M, M)  matrice de covariance de prévision
    """
    M = len(TICKERS)
    K = len(TV_FEATURES)

    alpha_mean = chains["alpha"].mean(axis=0)       # (M,)
    gamma_mean = chains["gamma"].mean(axis=0)       # (M, p)
    m_T        = chains["beta"].mean(axis=0)[-1]    # (M, K) — dernier état d'entraînement
    Omega_mean = chains["Omega"].mean(axis=0)       # (M, M)

    # ---- Point forecast (éq. 19) ----
    sent   = np.einsum('mk,mk->m', X_new, m_T)          # (M,)
    static = np.einsum('mp,mp->m', Z_new, gamma_mean)    # (M,)
    mu     = alpha_mean + sent + static                  # (M,)

    # ---- Forecast covariance (éq. 20) ----
    # F_{T+N} : vraie matrice bloc-diagonale (M, M*K), un bloc diag(X_new[:,k]) par feature k
    F = np.zeros((M, M * K))
    for k in range(K):
        F[:, k*M:(k+1)*M] = np.diag(X_new[:, k])

    # C_N : covariance des états à l'horizon N — bloc-diagonale (M*K, M*K),
    # chaque bloc k = N * Sigma_hat (même Sigma_hat partagée à travers les features,
    # cohérent avec le sampler qui suppose une structure Omega/Sigma commune)
    C_N = np.zeros((M * K, M * K))
    for k in range(K):
        C_N[k*M:(k+1)*M, k*M:(k+1)*M] = N * Sigma_hat

    # T*Sigma : accumulation de l'incertitude sur tout l'historique d'entraînement
    T_train = chains["beta"].shape[1]
    Sigma_tv = np.zeros((M * K, M * K))
    for k in range(K):
        Sigma_tv[k*M:(k+1)*M, k*M:(k+1)*M] = T_train * Sigma_hat

    Sigma = F @ (C_N + Sigma_tv) @ F.T + Omega_mean    # (M, M)

    return mu, Sigma


########### MAIN ##########
if __name__ == "__main__":

    for target in ['ret1']:
        print(f"TARGET: {target}")

        Y, X, Z, dates = build_panel(df, target, max_T=1258)

        # 1. Définition du split 80/20
        split_idx = int(len(Y) * 0.8)
        Y_train, X_train, Z_train = Y[:split_idx], X[:split_idx], Z[:split_idx]
        Y_test, X_test, Z_test, dates_test = Y[split_idx:], X[split_idx:], Z[split_idx:], dates[split_idx:]

        print(f"Y (train): {Y_train.shape}")
        print(f"Y (test): {Y_test.shape}")

        # 2. Exécution du Gibbs sampler uniquement sur le Train
        chains_list = []
        for chain_id, seed in enumerate([42, 123]):
            print(f"\nChain {chain_id + 1}/2 — seed={seed}")
            chain = run_gibbs(Y_train, X_train, Z_train, n_iter=N_ITER, burn_in=BURN_IN, delta=DELTA, seed=seed)
            chains_list.append(chain)

        # 3. Diagnostic Gelman-Rubin
        print("\nGelman-Rubin R-hat for alpha (Train):")
        for j, ticker in enumerate(TICKERS):
            rhat = gelman_rubin([c["alpha"][:, j] for c in chains_list])
            print(f"  {ticker}: R-hat = {rhat:.4f}")

        Sigma_hat = estimate_Sigma_post_gibbs([c["beta"] for c in chains_list])
        print(f"\nSigma_hat (post-Gibbs) diag: {np.diag(Sigma_hat)}")

        # 4. Combinaison des chaînes
        combined = {
            key: np.concatenate([c[key] for c in chains_list], axis=0)
            for key in chains_list[0]
        }

        # 5. Extraction des paramètres postérieurs moyens
        alpha_hat = combined["alpha"].mean(axis=0)        # (M,)
        gamma_hat = combined["gamma"].mean(axis=0)        # (M, p)
        m_T = combined["beta"].mean(axis=0)[-1]           # (M, K) - Dernier état d'entraînement

        # 6. Prédiction OOS (Application de l'équation 21 du papier)
        Y_pred = (alpha_hat[None, :]
                  + np.einsum('tmk,mk->tm', X_test, m_T)
                  + np.einsum('tmp,mp->tm', Z_test, gamma_hat))

        # OOS evaluation
        mae_oos = np.abs(Y_test - Y_pred).mean(axis=0)

        print(f"\nOut-of-sample MAE per stock ({target}):")
        for j, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {mae_oos[j]:.6f}")

        hit_rate = np.mean(np.sign(Y_pred) == np.sign(Y_test), axis=0)

        print("\nDirectional Accuracy (Hit Rate):")
        for j, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {hit_rate[j]:.4f}")

        strat_returns = np.sign(Y_pred) * Y_test
        cum_pnl = strat_returns.sum(axis=0)

        print("\nCumulative OOS P&L:")
        for j, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {cum_pnl[j]:.4f}")

        mse_model = np.sum((Y_test - Y_pred)**2, axis=0)
        mse_naive = np.sum(Y_test**2, axis=0)
        r2_oos = 1 - (mse_model / mse_naive)

        print("\nOut-of-Sample R-squared (R2 OOS):")
        for j, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {r2_oos[j]:.6f}")

TARGET: ret1
Y (train): (80, 6)
Y (test): (20, 6)

Chain 1/2 — seed=42
  [Numba] Compiling JIT kernels on first iteration...
  Iteration 500/2000
  Iteration 1000/2000
  Iteration 1500/2000
  Iteration 2000/2000

Chain 2/2 — seed=123
  [Numba] Compiling JIT kernels on first iteration...
  Iteration 500/2000
  Iteration 1000/2000
  Iteration 1500/2000
  Iteration 2000/2000

Gelman-Rubin R-hat for alpha (Train):
  AAPL: R-hat = 1.0032
  AMZN: R-hat = 1.0521
  GOOG: R-hat = 1.0038
  GOOGL: R-hat = 1.1288
  MSFT: R-hat = 1.5518
  TSLA: R-hat = 1.0119

Sigma_hat (post-Gibbs) diag: [0.00020371 0.0002373  0.00034959 0.00059137 0.00019787 0.00061284]

Out-of-sample MAE per stock (ret1):
  AAPL: 0.011499
  AMZN: 0.006095
  GOOG: 0.007225
  GOOGL: 0.007119
  MSFT: 0.015566
  TSLA: 0.015454

Directional Accuracy (Hit Rate):
  AAPL: 0.5000
  AMZN: 0.5500
  GOOG: 0.4500
  GOOGL: 0.4000
  MSFT: 0.6000
  TSLA: 0.5000

Cumulative OOS P&L:
  AAPL: 0.0040
  AMZN: -0.0053
  GOOG: 0.0108
  GOOGL: 0.0056
 

We set a simpliefid model running on half on the sample, with 2000 and 500 settings, and fill of mising values with an EWMA on past 5 windows with a 0.5 delta. We itterate on all individual tweet sentiment method combination (15), all headlines sentiment combinations (15) as well as all tweeter and headlines sentiment combination combos (15*15) for all three averaging method. We also test tweets using only overnight sentiment (from t market close to t+1 market open).

We notice that the log-weighted average on tweet metrics performs better than the two other averaging methods. By lowest R2 (still negative), the top 5 models are: 
- T=VADER | HL=RoBERTa+BoW+TBlob+VADER
- T=VADER | HL=VADER
- T=VADER | HL=BoW+TBlob+VADER
- overnight T=VADER
- overnight T=TBlob+VADER

For highest hit-rate, the top 5 is: 
- HL=VADER 
- overnight T=VADER
- T=VADER
- overnight T=TBlob
- T=BoW+VADER | HL=VADER	

For combining highest hit-rate and highest-r^2, the top 5 is: 
- T=VADER
- HL=VADER
- overnight T=VADER
- overnight T=TBlob
- T=VADER | HL=BoW+VADER

We hence shortlist the 10 following models, all using the log-weighted average: 
- T=VADER | HL=RoBERTa+BoW+TBlob+VADER
- T=VADER | HL=VADER
- T=VADER | HL=BoW+TBlob+VADER
- overnight T=VADER
- overnight T=TBlob+VADER
- HL=VADER
- T=VADER
- overnight T=TBlob
- T=BoW+VADER | HL=VADER
- T=VADER | HL=BoW+VADER
- overnight V2 T=BoW
- overnight V2 T=BoW+VADER